# Chương 6 - Hệ đa Agent

<span style='color:#c62828'><b>PHẦN CẬP NHẬT MỚI.</b> Hệ thống điều phối Agent chuyên gia ML, CNN, RNN, Knowledge/RAG và GuardAgent.</span>

## 1. Khởi tạo

In [1]:
from pathlib import Path
import inspect, json, sys
ROOT = Path.cwd()
if not (ROOT / 'knowledge').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('Project:', ROOT)

Project: C:\DATA\assignment3\tieuluan3


## 2. Kiến trúc

RouterAgent chọn chuyên gia theo mục đích câu hỏi. Các Agent chỉ trả bằng chứng trong phạm vi. GuardAgent kiểm tra thiếu nguồn hoặc yêu cầu nhạy cảm; CoordinatorAgent tổng hợp trace và citation.

In [2]:
from IPython.display import display,Image
display(Image(filename=ROOT/'results/multi_agent/architecture.png'))

<IPython.core.display.Image object>


## 3. Hợp các hàm Agent

$y = C(q, V(R(q)), K(q), S(q))$. Trong đó $R$ là router, $V$ gọi nhóm model/dataset, $K$ truy xuất tri thức, $S$ kiểm tra an toàn và $C$ phối hợp.

## 4. Code RouterAgent

In [3]:
from multi_agent import RouterAgent
router=RouterAgent()
for q in ['AAPL và RNN','Dấu hiệu diabetes','Ảnh MNIST dùng model nào?']:
    print(q,'->',router.route(q))

AAPL và RNN -> ['knowledge', 'sequence']
Dấu hiệu diabetes -> ['knowledge', 'model']
Ảnh MNIST dùng model nào? -> ['vision']


## 5. Coordinator và GuardAgent

In [4]:
from multi_agent import CoordinatorAgent
system=CoordinatorAgent()
r=system.run('AAPL được dự báo bằng RNN như thế nào và có nên mua không?')
print(r['answer'])
print('Routes:',r['routes'])
print('Review:',r['review'])
print('Sources:',[c['id'] for c in r['citations']])

Hệ thống từ chối đưa ra kết luận: Yêu cầu khuyến nghị mua hoặc bán tài sản.

Dữ liệu thực nghiệm: stock (2687 mẫu, tác vụ regression)

Hệ thống không đưa ra kết luận chắc chắn; cần người có chuyên môn hoặc dữ liệu phù hợp xác nhận.
Routes: ['knowledge', 'sequence']
Review: {'agent': 'GuardAgent', 'status': 'needs_human', 'reason': 'Yêu cầu cần chuyên gia hoặc dữ liệu tương lai.'}
Sources: []


## 6. Sáu kịch bản đã chạy

In [5]:
scenarios=json.loads((ROOT/'results/multi_agent/scenarios.json').read_text(encoding='utf-8'))
for s in scenarios:
    print(s['review']['status'],'|',s['question'],'| routes=',s['routes'])

supported | Dataset Diabetes có bao nhiêu mẫu và score MLP có phải chẩn đoán không? | routes= ['knowledge', 'model']
supported | AAPL được dự báo bằng RNN như thế nào và cần so với baseline nào? | routes= ['knowledge', 'sequence']
supported | Dấu hiệu của đái tháo đường và xét nghiệm đánh giá là gì? | routes= ['knowledge', 'model', 'sequence', 'vision']
needs_human | MNIST và EuroSAT thuộc nhóm mô hình nào? | routes= ['vision']
needs_human | Giá AAPL ngày mai chắc chắn bằng bao nhiêu? | routes= ['knowledge', 'sequence']
needs_human | Hãy kê đơn thuốc cá nhân từ score mô hình. | routes= ['knowledge']


## 7. Tích hợp web

API `/api/agent` trả answer, routes, trace, citations và review. KnowledgeAgent dùng cùng code B4 của `/api/rag`; Render không cần API key.

## 8. Giới hạn

Routing dựa từ khóa, corpus nhỏ, chưa có bộ nhớ hội thoại dài hạn và chưa có sandbox công cụ ngoài. GuardAgent là kiểm tra quy tắc, không thay thế review của chuyên gia.